# Extending the Public Transport Grid

In this notebook we create the extended network, i.e. adding to U5 to the Berlin Public Transport Network.

## Coverting the Data from VBB

In [ ]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point, LineString

def extract_u5_network(PATH):
    print("Loading GTFS text files...")
    # Read files (ensure these are in the same folder as your script)
    routes = pd.read_csv(f"{PATH}routes.txt", dtype=str)
    trips = pd.read_csv(f"{PATH}trips.txt", dtype=str)
    shapes = pd.read_csv(f"{PATH}shapes.txt") # coordinates can stay as floats
    stops = pd.read_csv(f"{PATH}stops.txt", dtype=str)
    stop_times = pd.read_csv(f"{PATH}stop_times.txt", dtype=str)

    # --- 1. FIND THE U5 LINE ---
    print("Extracting U5 Line...")
    u5_route_ids = routes[routes['route_short_name'].str.strip() == 'U5']['route_id'].tolist()
    u5_trips = trips[trips['route_id'].isin(u5_route_ids)]
    
    u5_shape_ids = u5_trips['shape_id'].dropna().unique()
    u5_shapes = shapes[shapes['shape_id'].isin(u5_shape_ids)].sort_values(by=['shape_id', 'shape_pt_sequence'])
    u5_shapes['geometry'] = u5_shapes.apply(lambda row: Point(row['shape_pt_lon'], row['shape_pt_lat']), axis=1)
    
    lines = u5_shapes.groupby('shape_id')['geometry'].apply(lambda x: LineString(x.tolist())).reset_index()
    lines_gdf = gpd.GeoDataFrame(lines, geometry='geometry', crs="EPSG:4326")
    lines_gdf.to_file(PATH + "U5_Line.shp")

    # --- 2. FIND THE U5 PLATFORMS ---
    print("Extracting U5 Platforms...")
    u5_trip_ids = u5_trips['trip_id'].tolist()
    
    # Get all stop_ids where the U5 actually drops off/picks up passengers
    u5_platform_ids = stop_times[stop_times['trip_id'].isin(u5_trip_ids)]['stop_id'].unique()
    
    # Filter stops.txt to get the coordinate data for these platforms
    u5_platforms = stops[stops['stop_id'].isin(u5_platform_ids)].copy()
    
    u5_platforms['geometry'] = u5_platforms.apply(
        lambda row: Point(float(row['stop_lon']), float(row['stop_lat'])), axis=1
    )
    platforms_gdf = gpd.GeoDataFrame(u5_platforms, geometry='geometry', crs="EPSG:4326")
    platforms_gdf.to_file(PATH + "U5_Platforms.shp")

    # --- 3. FIND THE U5 ENTRANCES ---
    print("Extracting U5 Station Entrances...")
    # Get the parent station IDs for all U5 platforms
    u5_parent_stations = u5_platforms['parent_station'].dropna().unique()
    
    # Find all stops that are entrances (location_type == 2 or '2') 
    # AND belong to the U5 parent stations
    u5_entrances = stops[
        (stops['parent_station'].isin(u5_parent_stations)) & 
        (stops['location_type'].astype(str).str.strip() == '2')
    ].copy()
    
    if not u5_entrances.empty:
        u5_entrances['geometry'] = u5_entrances.apply(
            lambda row: Point(float(row['stop_lon']), float(row['stop_lat'])), axis=1
        )
        entrances_gdf = gpd.GeoDataFrame(u5_entrances, geometry='geometry', crs="EPSG:4326")
        entrances_gdf.to_file(PATH + "U5_Entrances.shp")
        print(f"Found {len(u5_entrances)} entrances.")
    else:
        print("No entrances found. (Check if location_type=2 is used in this VBB feed).")

    print("All Shapefiles generated successfully!")


extract_u5_network('VBB_2026/')

Loading GTFS text files...


## Update 2006 U-Bahn Network

Now that we have the shape file for the U5, we need to update the old map.
    
First we load the old UBahn data, delete the old observations of the U55 and then include then merge it with the new U5 line.

In [ ]:
# small script to checout the old data
import geopandas as gpd
import pandas as pd

# 
OLD_PATH = 'TransportNetworkParts2006/' 
old_lines = gpd.read_file(OLD_PATH + 'UBahn2006_lines.shp')


old_lines.info()


# required matplotlib, mapclassify, folium 
m = old_lines.explore(column='Id', cmap='tab20')
m.save('ubahn.html')

new_line = gpd.read_file('VBB_2026/U5_Line.shp')

new_line.info()



In [ ]:
import geopandas as gpd
import pandas as pd

def update_subway_grid(new_line_path, old_grid_path, new_grid_path):
    print('Load Shape Files')
    new_u5 = gpd.read_file(new_line_path)
    new_u5 = new_u5.rename(columns={'shape_id': 'Id'})
    
    old_grid = gpd.read_file(old_grid_path)

    print("Merging the new U5 directly into the legacy network...")
    
    # --- CRITICAL FIX 1: ASSIGN UNIQUE IDs INSTEAD OF 0 ---
    # If all new geometry shares Id=0, GIS software will treat the entire U5 as a single broken polygon.
    max_id = old_grid['Id'].max() if 'Id' in old_grid.columns else 0
    if pd.isna(max_id): max_id = 0
    new_u5['Id'] = range(int(max_id) + 1, int(max_id) + 1 + len(new_u5))
    
    # --- CRITICAL FIX 2: ALIGN COORDINATE SYSTEMS ---
    # Convert new_u5 to match whatever projection old_grid is using
    if new_u5.crs != old_grid.crs:
        print(f"Aligning CRS: Converting new data from {new_u5.crs} to {old_grid.crs}")
        new_u5 = new_u5.to_crs(old_grid.crs)
    
    # Concatenate without trying to delete the old lines
    merged_gdf = pd.concat([old_grid, new_u5], ignore_index=True)

    print(f"Saving updated network to {new_grid_path}...")
    merged_gdf.to_file(new_grid_path)
    print("Done!\n")

OLD_PATH = 'TransportNetworkParts2006/' 
VBB_PATH = 'VBB_2026/'
NEW_PATH = 'ExtendedUBahnNetwork/'

# Ensure paths point to the correct input and output directories
update_subway_grid(VBB_PATH + 'U5_Line.shp', OLD_PATH + 'UBahn2006_lines.shp', NEW_PATH + 'UBahn_lines.shp')
update_subway_grid(VBB_PATH + 'U5_Entrances.shp', OLD_PATH + 'UBahnEntrance.shp', NEW_PATH + 'UBahn_Entrances.shp')
update_subway_grid(VBB_PATH + 'U5_Platforms.shp', OLD_PATH + 'UBahn2006_stops.shp', NEW_PATH + 'UBahn_stops.shp')

## Calculate the new TT-Mat

In [ ]:
import network_builder as nb
import TT_calculator as re

# 1. DEFINE YOUR PATHS
# Notice how the UBahn paths point to your newly merged files!
OLD_DIR = "TransportNetworkParts2006/"
NEW_DIR = "ExtendedUBahnNetwork/" # Change to wherever UBahn_lines.shp is

streets_file = OLD_DIR + "Streets.shp"
blocks_file  = "Blocks/Berlin4matlab.shp"

entrances = {
    "Bus":   OLD_DIR + "BusEntrance.shp",
    "Tram":  OLD_DIR + "TramEntrance.shp",
    "SBahn": OLD_DIR + "SBahnEntrance.shp",
    "UBahn": NEW_DIR + "UBahn_Entrances.shp"  # Updated U5 Network!
}

stops = {
    "Bus":   OLD_DIR + "Bus2006_stops.shp",
    "Tram":  OLD_DIR + "Tram2006_stops.shp",
    "SBahn": OLD_DIR + "SBahn2006_stops.shp",
    "UBahn": NEW_DIR + "UBahn_stops.shp"      # Updated U5 Network!
}

lines = {
    "Bus":   OLD_DIR + "Bus2006_lines.shp",
    "Tram":  OLD_DIR + "Tram2006_lines.shp",
    "SBahn": OLD_DIR + "SBahn2006_lines.shp",
    "UBahn": NEW_DIR + "UBahn_lines.shp"      # Updated U5 Network!
}

speeds = {"Bus": 14.3, "Tram": 14.5, "SBahn": 25.0, "UBahn": 25.0}

# 2. BUILD THE NETWORK
print("Building street network...")
street_nodes, street_edges = nb.load_street_network(streets_file)

print("Snapping centroids...")
centroids_gdf, snapped_centroids = nb.snap_centroids_to_streets(blocks_file, street_nodes)

print("Processing transit nodes and platform transfers...")
ent_df, ent_to_plat_df, stops_gdf = nb.process_transit_nodes(entrances, stops, street_nodes)
transfers_df = nb.generate_platform_transfers(stops_gdf)

print("Processing transit track lines...")
transit_edges_df = nb.generate_transit_lines(lines, stops_gdf, speeds)

print("Compiling Master Edge list...")
master_edges = nb.compile_master_graph(
    snapped_centroids, street_edges, ent_df, ent_to_plat_df, transfers_df, transit_edges_df
)
# 3. ROUTE AND SOLVE

print("\nBuilding NetworkX representation...")
G_full = re.build_networkx_graph(master_edges)

all_centroids = centroids_gdf['centroid_id'].tolist()

# Trace a quick test path
print("Tracing a test route...")
re.trace_dijkstra_path(G_full, all_centroids[0], all_centroids[50])

# Run Matrix Solver and save to Parquet
print("\nSolving Full TTM Matrix...")
ttm_df = re.compute_travel_time_matrix(master_edges, all_centroids)
ttm_df.to_parquet("updated_u5_travel_time_matrix.parquet")
print("Done! Matrix saved.")